# Sponge Metrics: Urban Flood Susceptibility & SuDS Evaluation
## Grid-Based Framework for Oxford, UK

## Objectives

* Develop a grid-based urban flood susceptibility framework for Oxford, United Kingdom.
* Estimate surface runoff and flood susceptibility under extreme rainfall conditions.
* Evaluate SuDS intervention impacts on runoff reduction and flood resilience.
* Create spatial flood hotspot analysis and climate scenario testing.

## Inputs

* Multi-timescale rainfall forcing system derived from 15-minute observations.
* Event-scale rainfall layers: max_15min_rain, max_1h_rain, storm_event_total_rain.
* Antecedent rainfall layers: rain_3d, rain_5d, rain_7d.
* Extremity indicators: rain_95p_flag, extreme_event_flag.
* Topographic data (elevation, slope per 1 km grid cell).
* Urban surface & land use data (imperviousness %, green space %, building density).
* Floodplain and river proximity flags for spatial context.
* SuDS scenario definitions (rain gardens, permeable paving, detention basins, swales, green roofs).

## Outputs

* Grid-based spatial features DataFrame (elevation, slope, imperviousness, floodplain, river proximity).
* Rainfall forcing tables for event, antecedent, and stress layers.
* Runoff coefficient and effective rainfall outputs.
* Flood susceptibility classifications (Low, Medium, High risk).
* Climate scenario outputs (10-yr, 30-yr, 100-yr rainfall events).
* SuDS comparison maps (existing vs. with interventions).
* Streamlit dashboard visualizations and KPIs.

## Libraries Import

In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import math
from scipy import stats
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

print("All libraries imported successfully.")


All libraries imported successfully.


## Set project root for file I/O

* The notebook should resolve the project root once at startup, then use explicit paths from that root for all reads and writes

The notebook resolves the project root once at startup.
* We access the current directory with Path.cwd()

In [3]:
current_dir = Path.cwd()
for candidate in [current_dir, *current_dir.parents]:
    if (candidate / "data").exists():
        project_root = candidate
        break
else:
    project_root = current_dir

raw_data_dir = project_root / "data" / "raw"
processed_data_dir = project_root / "data" / "processed"
engineered_data_dir = project_root / "data" / "engineered"
current_dir = str(project_root)
current_dir

'c:\\Users\\Home\\OneDrive\\Python Projects\\sponge-metrics'

We want to detect the project root once and keep it fixed for the rest of the notebook.
* os.path.dirname() is no longer needed here because file paths are built explicitly from the project root.
* os.chdir() is not used.

In [4]:
print(f"Project root detected: {project_root}")

Project root detected: c:\Users\Home\OneDrive\Python Projects\sponge-metrics


Confirm the detected project root

In [5]:
current_dir = str(project_root)
current_dir

'c:\\Users\\Home\\OneDrive\\Python Projects\\sponge-metrics'

## Section 1: Spatial Grid Setup

### 1.1 Oxford Grid Cell Definition
A 1 km × 1 km grid framework covering Oxford, United Kingdom was selected to balance spatial representation, computational efficiency, and data availability. Given the city-scale objective of identifying relative surface-water flood susceptibility rather than property-scale inundation, the chosen resolution was considered appropriate.

In [6]:
# Bounding box (Oxford, UK as study area)
lat_min, lat_max = 51.72, 51.80   
lon_min, lon_max = -1.3, -1.2

# Grid size in meters
grid_size_m = 1000  # convert 1 km to m

# Approximate degree step sizes for the target grid size
mean_lat = (lat_min + lat_max) / 2
dlat = grid_size_m / 111320   # 1 degree of latitude is approximately 111.32 km
dlon = grid_size_m / (111320 * math.cos(math.radians(mean_lat)))  

rows = []  # list to hold grid cell information
location_idx = 1 

lat = lat_min  
while lat < lat_max:
    next_lat = min(lat + dlat, lat_max)  

    lon = lon_min
    while lon < lon_max:
        next_lon = min(lon + dlon, lon_max) 

        rows.append(
            {
                "location_id": f"OXF_{location_idx:02d}", 
                "lat_min": round(lat, 6),    # round to 6 decimal places for better readability
                "lat_max": round(next_lat, 6),
                "lon_min": round(lon, 6),
                "lon_max": round(next_lon, 6),
                "centroid_lat": round((lat + next_lat) / 2, 6),
                "centroid_lon": round((lon + next_lon) / 2, 6),
            }
        )

        location_idx += 1     # increment location index for the next grid cell
        lon = next_lon

    lat = next_lat

df = pd.DataFrame(rows) # create DataFrame from the list of grid cells
print(df.shape)  # check the shape of the grid DataFrame to confirm the number of cells created
df.head()  # display the first few rows of the grid DataFrame to verify the output

(63, 7)


,location_id,lat_min,lat_max,lon_min,lon_max,centroid_lat,centroid_lon
0,OXF_01,51.72,51.728983,-1.300000,-1.285487,51.724492,-1.292743
1,OXF_02,51.72,51.728983,-1.285487,-1.270973,51.724492,-1.278230
2,OXF_03,51.72,51.728983,-1.270973,-1.256460,51.724492,-1.263717
3,OXF_04,51.72,51.728983,-1.256460,-1.241947,51.724492,-1.249203
4,OXF_05,51.72,51.728983,-1.241947,-1.227433,51.724492,-1.234690


### 1.2 Topographic Data: Elevation & Slope
Assign elevation per grid cell using information seen on CalcMaps and OS Maps. This is then used to calculate slope (m/m) and derive slope percentage. Slope is used in runoff coefficient model and water accumulation tendency.

In [7]:
# Assign elevations for the current 9x7 grid
elevations = [
    136, 103, 99, 58, 81, 75, 66,
    117, 102, 59, 58, 60, 64, 78,
    109, 59, 58, 56, 66, 71, 88,
    62, 59, 62, 62, 66, 93, 97,
    58, 57, 67, 61, 68, 102, 103,
    59, 60, 69, 58, 61, 74, 71,
    59, 60, 67, 57, 62, 65, 86,
    59, 74, 69, 58, 58, 96, 115,
    62, 75, 66, 58, 59, 70, 82,
 ]

nrows = 9
ncols = 7

if len(elevations) != len(df):
    raise ValueError(f"Expected {len(df)} elevation values, got {len(elevations)}")

df["elevation_m"] = elevations   # add elevation column (row-major order for current grid)

# Compute actual slope (m/m)
slope = []
for r in range(nrows):
    for c in range(ncols):
        cur = elevations[r * ncols + c]
        diffs = []
        # Left, Right, Up, Down (if present)
        if c > 0:
            diffs.append(abs(cur - elevations[r * ncols + (c - 1)]))  
        if c < ncols - 1:
            diffs.append(abs(cur - elevations[r * ncols + (c + 1)]))
        if r > 0:
            diffs.append(abs(cur - elevations[(r - 1) * ncols + c]))
        if r < nrows - 1:
            diffs.append(abs(cur - elevations[(r + 1) * ncols + c]))

        avg_diff = sum(diffs) / len(diffs)
        slope_val = round(avg_diff / grid_size_m, 4)  # slope in m/m, rounded
        slope.append(slope_val)

df["slope"] = slope
df["slope_pct"] = [round(s * 100, 1) for s in slope]  # add slope percent column

df.head()  # display the first few rows of the final DataFrame with elevations and slopes

,location_id,lat_min,lat_max,lon_min,lon_max,centroid_lat,centroid_lon,elevation_m,slope,slope_pct
0,OXF_01,51.72,51.728983,-1.300000,-1.285487,51.724492,-1.292743,136,0.0260,2.6
1,OXF_02,51.72,51.728983,-1.285487,-1.270973,51.724492,-1.278230,103,0.0127,1.3
2,OXF_03,51.72,51.728983,-1.270973,-1.256460,51.724492,-1.263717,99,0.0283,2.8
3,OXF_04,51.72,51.728983,-1.256460,-1.241947,51.724492,-1.249203,58,0.0213,2.1
4,OXF_05,51.72,51.728983,-1.241947,-1.227433,51.724492,-1.234690,81,0.0167,1.7


### 1.3 Urban Surface Characteristics
Characterize urban surface conditions across the study area using satellite imagery interpretation. Each grid cell was assigned representative imperviousness, vegetation cover, and building-density values based on its dominant land-cover type.

In [8]:
surface_specs = [
    (["OXF_01"], "Suburban Residential", 40, 60, 0.35),
    (["OXF_02", "OXF_60"], "Green Space", 10, 90, 0.05),
    (["OXF_03", "OXF_09", "OXF_17"], "Green Space", 25, 75, 0.15),
    (["OXF_04", "OXF_10", "OXF_55"], "Suburban Residential", 45, 55, 0.45),
    (["OXF_05", "OXF_08", "OXF_12", "OXF_21", "OXF_26", "OXF_44", "OXF_50"], "Dense Residential", 68, 32, 0.68),
    (["OXF_06", "OXF_11", "OXF_13", "OXF_20", "OXF_27", "OXF_28"], "Dense Residential", 72, 28, 0.72),
    (["OXF_07", "OXF_19", "OXF_23", "OXF_34", "OXF_40"], "Dense Residential", 78, 22, 0.78),
    (["OXF_14"], "Urban Core", 80, 20, 0.80),
    (["OXF_15", "OXF_16", "OXF_22"], "Suburban Residential", 60, 40, 0.55),
    (["OXF_18", "OXF_48", "OXF_49"], "Green Space", 8, 92, 0.03),
    (["OXF_24", "OXF_25"], "Urban Core", 88, 12, 0.90),
    (["OXF_29", "OXF_32", "OXF_36", "OXF_39", "OXF_46", "OXF_61"], "Green Space", 12, 88, 0.08),
    (["OXF_30", "OXF_37", "OXF_43", "OXF_53", "OXF_54", "OXF_57"], "Green Space", 25, 75, 0.25),
    (["OXF_31"], "Urban Core", 85, 15, 0.85),
    (["OXF_33", "OXF_35", "OXF_38", "OXF_41", "OXF_42", "OXF_45", "OXF_52"], "Dense Residential", 75, 25, 0.73),
    (["OXF_47", "OXF_51"], "Dense Residential", 67, 33, 0.67),
    (["OXF_56", "OXF_59", "OXF_62", "OXF_63"], "Green Space", 15, 85, 0.15),
    (["OXF_58"], "Suburban Residential", 45, 55, 0.45),
]

surface_lookup = {}
for location_ids, area_type, impervious_surface_pct, green_space_pct, building_density in surface_specs:
    for location_id in location_ids:
        surface_lookup[location_id] = {
            "area_type": area_type,
            "impervious_surface_pct": impervious_surface_pct,
            "green_space_pct": green_space_pct,
            "building_density": building_density,
        }

missing_locations = sorted(set(df["location_id"]) - set(surface_lookup))
if missing_locations:
    raise ValueError(f"Missing surface attributes for: {missing_locations}")

df = df.assign(
    **pd.DataFrame(df["location_id"].map(surface_lookup).tolist(), index=df.index)
)

df.head()

,location_id,lat_min,lat_max,lon_min,lon_max,centroid_lat,centroid_lon,elevation_m,slope,slope_pct,area_type,impervious_surface_pct,green_space_pct,building_density
0,OXF_01,51.72,51.728983,-1.300000,-1.285487,51.724492,-1.292743,136,0.0260,2.6,Suburban Residential,40,60,0.35
1,OXF_02,51.72,51.728983,-1.285487,-1.270973,51.724492,-1.278230,103,0.0127,1.3,Green Space,10,90,0.05
2,OXF_03,51.72,51.728983,-1.270973,-1.256460,51.724492,-1.263717,99,0.0283,2.8,Green Space,25,75,0.15
3,OXF_04,51.72,51.728983,-1.256460,-1.241947,51.724492,-1.249203,58,0.0213,2.1,Suburban Residential,45,55,0.45
4,OXF_05,51.72,51.728983,-1.241947,-1.227433,51.724492,-1.234690,81,0.0167,1.7,Dense Residential,68,32,0.68


### 1.4 Hydrological Features: River Proximity and Floodplain Indicators
River proximity and floodplain indicators were included as supplementary hydrological variables based on visual interpretation from satellite imagery and local floodplain information. Although, they were not incorporated into the final susceptibility model, as the framework focuses specifically on urban surface-water flooding rather than fluvial flooding.

In [9]:
river_proximity = {
    "OXF_04",
    "OXF_05",
    "OXF_11",
    "OXF_12",
    "OXF_17",
    "OXF_18",
    "OXF_22",
    "OXF_23",
    "OXF_24",
    "OXF_25",
    "OXF_29",
    "OXF_30",
    "OXF_32",
    "OXF_36",
    "OXF_37",
    "OXF_38",
    "OXF_39",
    "OXF_43",
    "OXF_44",
    "OXF_45",
    "OXF_46",
    "OXF_50",
    "OXF_51",
    "OXF_52",
    "OXF_53",
    "OXF_54",
    "OXF_58",
    "OXF_59",
    "OXF_60",
    "OXF_61",
}

floodplain_locations = {
    "OXF_18",
    "OXF_29",
    "OXF_30",
    "OXF_32",
    "OXF_36",
    "OXF_37",
    "OXF_39",
    "OXF_43",
    "OXF_46",
    "OXF_53",
    "OXF_54",
    "OXF_59",
    "OXF_60",
    "OXF_61",
}

df["river_proximity_flag"]  = df["location_id"].isin(river_proximity).astype(int)

df["floodplain_flag"] = df["location_id"].isin(floodplain_locations).astype(int)


df.head()

,location_id,lat_min,lat_max,lon_min,lon_max,centroid_lat,centroid_lon,elevation_m,slope,slope_pct,area_type,impervious_surface_pct,green_space_pct,building_density,river_proximity_flag,floodplain_flag
0,OXF_01,51.72,51.728983,-1.300000,-1.285487,51.724492,-1.292743,136,0.0260,2.6,Suburban Residential,40,60,0.35,0,0
1,OXF_02,51.72,51.728983,-1.285487,-1.270973,51.724492,-1.278230,103,0.0127,1.3,Green Space,10,90,0.05,0,0
2,OXF_03,51.72,51.728983,-1.270973,-1.256460,51.724492,-1.263717,99,0.0283,2.8,Green Space,25,75,0.15,0,0
3,OXF_04,51.72,51.728983,-1.256460,-1.241947,51.724492,-1.249203,58,0.0213,2.1,Suburban Residential,45,55,0.45,1,0
4,OXF_05,51.72,51.728983,-1.241947,-1.227433,51.724492,-1.234690,81,0.0167,1.7,Dense Residential,68,32,0.68,1,0


### 1.5 Hydrologic Soil Group (HSG)
Hydrologic Soil Groups were approximated using broad geological and environmental characteristics across Oxford. Cells located near river corridors were assigned HSG-B to represent relatively free-draining alluvial soils, while the remaining urban areas were assigned HSG-D to reflect the low-permeability clay-dominated soils common across much of Oxford. These classifications were later used in deriving Curve Number (CN) values for runoff estimation.

In [10]:
# Assign Hydrologic Soil Group (HSG) based on river proximity
df["hydrologic_soil_group"] = df["river_proximity_flag"].apply(
    lambda x: "B" if x == 1 else "D"   # assign HSG B if river is nearby, otherwise D
)

df.head()


,location_id,lat_min,lat_max,lon_min,lon_max,centroid_lat,centroid_lon,elevation_m,slope,slope_pct,area_type,impervious_surface_pct,green_space_pct,building_density,river_proximity_flag,floodplain_flag,hydrologic_soil_group
0,OXF_01,51.72,51.728983,-1.300000,-1.285487,51.724492,-1.292743,136,0.0260,2.6,Suburban Residential,40,60,0.35,0,0,D
1,OXF_02,51.72,51.728983,-1.285487,-1.270973,51.724492,-1.278230,103,0.0127,1.3,Green Space,10,90,0.05,0,0,D
2,OXF_03,51.72,51.728983,-1.270973,-1.256460,51.724492,-1.263717,99,0.0283,2.8,Green Space,25,75,0.15,0,0,D
3,OXF_04,51.72,51.728983,-1.256460,-1.241947,51.724492,-1.249203,58,0.0213,2.1,Suburban Residential,45,55,0.45,1,0,B
4,OXF_05,51.72,51.728983,-1.241947,-1.227433,51.724492,-1.234690,81,0.0167,1.7,Dense Residential,68,32,0.68,1,0,B


### 1.6 Spatial Features Output

In [11]:
raw_data_dir.mkdir(parents=True, exist_ok=True)
df.to_csv(raw_data_dir / "oxf_grid_features.csv", index=False)
print(f"✓ Saved: {raw_data_dir / 'oxf_grid_features.csv'}")

✓ Saved: c:\Users\Home\OneDrive\Python Projects\sponge-metrics\data\raw\oxf_grid_features.csv


## Section 2: Meteorological Data 

### 2.1 Rainfall Data Loading and Cleaning
Prepare 15-minute rainfall observations from Osney Lock (January 2000 – May 2026) for hydrological modelling through data cleaning, timestamp standardization, and handling of missing values.

In [12]:
# Load and prepare 15-minute rainfall observations
rain_15 = pd.read_csv(raw_data_dir / "rainfall_15mins.csv")

# Standardize data types
rain_15["dateTime"] = pd.to_datetime(   # convert dateTime to pandas datetime, coercing errors to NaT
    rain_15["dateTime"],
    errors="coerce"
)

rain_15["value"] = pd.to_numeric(       # convert rainfall values to numeric, coercing errors to NaN
    rain_15["value"],
    errors="coerce"
)

print("Initial dataset")
print(f"Records: {len(rain_15):,}")
print("\nMissing values:")
print(rain_15[["dateTime", "value"]].isna().sum())    # check for missing values in key columns

rain_15 = rain_15.dropna(subset=["dateTime"]).copy()  # remove records without valid timestamps

rain_15["value"] = rain_15["value"].fillna(0)         # fill missing rainfall values with 0

rain_15["date"] = rain_15["dateTime"].dt.floor("D")   # create daily identifier for subsequent aggregation

print("\nFirst five records:")
display(rain_15.head())

Initial dataset
Records: 926,205

Missing values:
dateTime        0
value       22859
dtype: int64

First five records:


,dateTime,date,value
0,2000-01-01 00:00:00,2000-01-01,0.2
1,2000-01-01 00:15:00,2000-01-01,0.0
2,2000-01-01 00:30:00,2000-01-01,0.0
3,2000-01-01 00:45:00,2000-01-01,0.0
4,2000-01-01 01:00:00,2000-01-01,0.0


* The rainfall archive contained complete temporal coverage between January 2000 and May 2026, with no missing calendar days. 
* Approximately 2.5% of 15-minute rainfall observations contained missing values. 
* These were replaced with 0 mm under the assumption that missing measurements primarily represented periods of no recorded rainfall, thereby preserving continuity of the rainfall time series for subsequent aggregation and hydrological analysis

### 2.2 Rainfall Data Quality Assessment
Assess the completeness, consistency, and statistical characteristics of the rainfall dataset to ensure its suitability for hydrological modelling and extreme rainfall analysis.

In [13]:
# Summary checks.
date_start = rain_15["date"].min()
date_end = rain_15["date"].max()

expected_days = (date_end - date_start).days + 1           # calculate expected number of days in the date range (inclusive)
observed_days = rain_15["date"].nunique()                  # count unique days in the cleaned dataset

print("\nCleaned dataset")
print(f"Records: {len(rain_15):,}")
print(f"Date range: {date_start.date()} to {date_end.date()}")
print(f"Observed days: {observed_days:,}")
print(f"Expected days: {expected_days:,}")

if observed_days != expected_days:
    print(
        f"Warning: {expected_days - observed_days:,} day(s) "  
        "are absent from the record."
    )

print("\nRainfall occurrence frequency:")
rainfall_frequency = (
    (rain_15["value"] > 0).mean() * 100
)
print(f"{rainfall_frequency:.2f}%")                        # percentage of records with non-zero rainfall

print("\nRainfall value summary (non-zero records):")
rain_15.loc[rain_15["value"] > 0, "value"].describe()      # summary statistics for non-zero rainfall values


Cleaned dataset
Records: 926,205
Date range: 2000-01-01 to 2026-05-31
Observed days: 9,648
Expected days: 9,648

Rainfall occurrence frequency:
5.54%

Rainfall value summary (non-zero records):


count    51352.00000
mean         0.32486
std          0.51220
min          0.01000
25%          0.12000
50%          0.20000
75%          0.40000
max         22.00000
Name: value, dtype: float64

* Rainfall was recorded in approximately 5.54% of all 15-minute intervals, confirming the expected sparsity of rainfall events.
* Positive rainfall observations exhibited a realistic right-skewed distribution, with a median intensity of 0.20 mm and a maximum recorded 15-minute rainfall of 22 mm.
* The dataset was therefore considered suitable for subsequent aggregation, hydrological modelling, and extreme-value analysis.

### 2.3 Daily Rainfall Aggregation
Transform 15-minute rainfall observations into a daily event-scale rainfall dataset, capturing total rainfall, short-duration intensity, and peak 1-hour rainfall to support hydrological modelling.

In [14]:
# Aggregate 15-minute rainfall observations into daily forcing variables.
rainfall_daily = (
    rain_15.groupby("date", as_index=False)
    .agg(
        max_15min_rain=("value", "max"),
        storm_event_total_rain=("value", "sum"),
    )
)

# Daily maximum rolling 1-hour rainfall intensity.
rainfall_daily["max_1h_rain"] = (
    rain_15.sort_values(["date", "dateTime"])
    .groupby("date")["value"]
    .apply(lambda x: x.rolling(4, min_periods=1).sum().max()) 
    .values
)

rainfall_daily[
    ["max_15min_rain", "max_1h_rain", "storm_event_total_rain"]  
] = rainfall_daily[
    ["max_15min_rain", "max_1h_rain", "storm_event_total_rain"]
].round(1)               # round rainfall values to 1 decimal place for better readability        

# Ensure continuous daily coverage.
calendar = pd.date_range(
    rainfall_daily["date"].min(),
    rainfall_daily["date"].max(),
    freq="D"
)

rainfall_daily = (
    rainfall_daily.set_index("date")
    .reindex(calendar)
    .fillna(0.0)
    .rename_axis("date")
    .reset_index()
)

# Time variables.
rainfall_daily["year"] = rainfall_daily["date"].dt.year
rainfall_daily["month"] = rainfall_daily["date"].dt.month
rainfall_daily["day"] = rainfall_daily["date"].dt.day

# Daily rainfall forcing used throughout the project.
rainfall_daily["rainfall_mm"] = rainfall_daily["storm_event_total_rain"]

print("✓ Daily rainfall aggregation complete")
print(f"Days available: {len(rainfall_daily):,}")

rainfall_daily.head()

✓ Daily rainfall aggregation complete
Days available: 9,648


,date,max_15min_rain,storm_event_total_rain,max_1h_rain,year,month,day,rainfall_mm
0,2000-01-01,0.2,0.2,0.2,2000,1,1,0.2
1,2000-01-02,0.2,0.2,0.2,2000,1,2,0.2
2,2000-01-03,0.8,7.6,2.6,2000,1,3,7.6
3,2000-01-04,0.8,1.0,1.0,2000,1,4,1.0
4,2000-01-05,0.0,0.0,0.0,2000,1,5,0.0


### 2.4 Rainfall Forcing Layers
Generate multi-timescale rainfall indicators, including antecedent rainfall accumulations, extreme rainfall flags, and seasonal classifications, to represent the temporal variability of rainfall conditions influencing flood generation processes.

#### 2.4.1 Antecedent Rainfall Conditions
Quantify short-term catchment wetness conditions using rolling rainfall accumulation over 3, 5, and 7-day windows.

In [15]:
rainfall_daily = pd.read_csv(processed_data_dir / "rainfall.csv", parse_dates=["date"])

# Antecedent rainfall (catchment wetness proxies)
rainfall_daily["rain_3d"] = rainfall_daily["storm_event_total_rain"].rolling(   
    window=3, min_periods=1
).sum().round(1)

rainfall_daily["rain_5d"] = rainfall_daily["storm_event_total_rain"].rolling(
    window=5, min_periods=1
).sum().round(1)

rainfall_daily["rain_7d"] = rainfall_daily["storm_event_total_rain"].rolling(
    window=7, min_periods=1
).sum().round(1)

print("Antecedent rainfall features added:")
print(rainfall_daily[["date", "rain_3d", "rain_5d", "rain_7d"]].head())

Antecedent rainfall features added:
        date  rain_3d  rain_5d  rain_7d
0 2000-01-01      0.2      0.2      0.2
1 2000-01-02      0.4      0.4      0.4
2 2000-01-03      8.0      8.0      8.0
3 2000-01-04      8.8      9.0      9.0
4 2000-01-05      8.6      9.0      9.0


#### 2.4.2 Rainfall Extremity Indicators
Identify extreme rainfall conditions using percentile-based thresholds and short-duration intensity metrics.

In [16]:
# Thresholds for extreme rainfall conditions
rain_95p_threshold = rainfall_daily["storm_event_total_rain"].quantile(0.95)   # 95th percentile of daily total rainfall as a threshold for extreme events
max_1h_threshold = rainfall_daily["max_1h_rain"].quantile(0.95)                # 95th percentile of maximum 1-hour rainfall as a threshold for extreme intensity events

# Extreme rainfall flag 
rainfall_daily["rain_95p_flag"] = (
    rainfall_daily["storm_event_total_rain"] >= rain_95p_threshold             
).astype(int)

rainfall_daily["extreme_event_flag"] = (                                  
    (rainfall_daily["storm_event_total_rain"] >= rain_95p_threshold) |     
    (rainfall_daily["max_1h_rain"] >= max_1h_threshold)                        # flag days where either daily total or max 1-hour rainfall exceeds their respective 95th percentile thresholds
).astype(int)

print("Extreme rainfall indicators added:")
print(rainfall_daily[[
    "date",
    "storm_event_total_rain",
    "max_1h_rain",
    "rain_95p_flag",
    "extreme_event_flag"
]].head())

Extreme rainfall indicators added:
        date  storm_event_total_rain  max_1h_rain  rain_95p_flag  \
0 2000-01-01                     0.2          0.2              0   
1 2000-01-02                     0.2          0.2              0   
2 2000-01-03                     7.6          2.6              0   
3 2000-01-04                     1.0          1.0              0   
4 2000-01-05                     0.0          0.0              0   

   extreme_event_flag  
0                   0  
1                   0  
2                   0  
3                   0  
4                   0  


#### 2.4.3 Seasonal Rainfall Classification
Introduce seasonal variability into the rainfall forcing system by distinguishing between wetter and drier hydrological periods.

In [17]:
rainfall_daily["season"] = rainfall_daily["month"].apply(
    lambda m: "growing" if 5 <= m <= 10 else "dormant"          # classify months May to October as growing season, others as dormant season
)

print("Seasonal classification added:")
print(rainfall_daily[["date", "month", "season"]].head())

print("\nSeason distribution:")
print(rainfall_daily["season"].value_counts())                   # count of records in each season category

Seasonal classification added:
        date  month   season
0 2000-01-01      1  dormant
1 2000-01-02      1  dormant
2 2000-01-03      1  dormant
3 2000-01-04      1  dormant
4 2000-01-05      1  dormant

Season distribution:
season
dormant    4833
growing    4815
Name: count, dtype: int64


### 2.5 Rainfall Dataset Export
Save the processed daily rainfall dataset as a reusable input for subsequent hydrological, validation, and flood-risk assessment tasks.

In [18]:
processed_data_dir.mkdir(parents=True, exist_ok=True)

rainfall_daily.to_csv(processed_data_dir / "rainfall.csv", index=False)

print(f"✓ Saved rainfall dataset: {processed_data_dir / 'rainfall.csv'}")
print(f"Columns saved ({len(rainfall_daily.columns)}): {rainfall_daily.columns.tolist()}")

✓ Saved rainfall dataset: c:\Users\Home\OneDrive\Python Projects\sponge-metrics\data\processed\rainfall.csv
Columns saved (14): ['date', 'max_15min_rain', 'storm_event_total_rain', 'max_1h_rain', 'year', 'month', 'day', 'rainfall_mm', 'rain_3d', 'rain_5d', 'rain_7d', 'rain_95p_flag', 'extreme_event_flag', 'season']


## Section 3: Master Dataset Integration

Integrate spatial and meteorological datasets into a unified spatio-temporal framework by associating each grid cell with daily rainfall conditions. This produces the master dataset used for subsequent hydrological modelling, flood susceptibility assessment, and validation. 

(cross-join: 63 cells × 9648 days = 607824 records)

In [19]:
master = df.merge(rainfall_daily, how="cross")             # create master dataset by performing a cross join between grid features and daily rainfall data

master.to_csv(
    processed_data_dir / "master_grid_dataset.csv",        # save the master dataset as CSV for easy inspection and GitHub visibility
    index=False
)

master.to_parquet(
    processed_data_dir / "master_grid_dataset.parquet",    # save the master dataset as Parquet for efficient storage and retrieval

    index=False
)

print("✓ Master dataset integration complete")
print(f"Master dataset shape: {master.shape}")             
print(f"Rows: {len(master):,}")                            # print the number of records in the master dataset
print(f"Columns: {len(master.columns)}")                   # print the number of columns in the master dataset

master.head()

✓ Master dataset integration complete
Master dataset shape: (607824, 31)
Rows: 607,824
Columns: 31


,location_id,lat_min,lat_max,lon_min,lon_max,centroid_lat,centroid_lon,elevation_m,slope,slope_pct,...,year,month,day,rainfall_mm,rain_3d,rain_5d,rain_7d,rain_95p_flag,extreme_event_flag,season
0,OXF_01,51.72,51.728983,-1.3,-1.285487,51.724492,-1.292743,136,0.026,2.6,...,2000,1,1,0.2,0.2,0.2,0.2,0,0,dormant
1,OXF_01,51.72,51.728983,-1.3,-1.285487,51.724492,-1.292743,136,0.026,2.6,...,2000,1,2,0.2,0.4,0.4,0.4,0,0,dormant
2,OXF_01,51.72,51.728983,-1.3,-1.285487,51.724492,-1.292743,136,0.026,2.6,...,2000,1,3,7.6,8.0,8.0,8.0,0,0,dormant
3,OXF_01,51.72,51.728983,-1.3,-1.285487,51.724492,-1.292743,136,0.026,2.6,...,2000,1,4,1.0,8.8,9.0,9.0,0,0,dormant
4,OXF_01,51.72,51.728983,-1.3,-1.285487,51.724492,-1.292743,136,0.026,2.6,...,2000,1,5,0.0,8.6,9.0,9.0,0,0,dormant


- The master dataset was exported in both CSV and Parquet formats. 
- The CSV version provides transparency and ease of inspection during development, while the Parquet version offers improved storage efficiency, faster loading times, and preservation of data types for subsequent modelling and application deployment.

## Section 4 Hydrological Modelling

### 4.1 Baseline Curve Number (CNII)

Establish a baseline Curve Number (CN) for each grid cell under average antecedent moisture conditions (AMC II).

1. Assign standard SCS Curve Numbers to:
   - Impervious surfaces (`CN = 98`)
   - Green space according to hydrologic soil group
2. Verify that impervious and green-space percentages sum to 100% for each grid cell
3. Compute a composite baseline Curve Number using area-weighted land-cover proportions:

$$
CN_{II}
=
\frac{
(\%Impervious \times CN_{impervious})
+
(\%Green \times CN_{green})
}
{100}
$$

The resulting `CN_II` represents the baseline runoff potential of each grid cell before antecedent moisture adjustments are applied.

In [ ]:
master = pd.read_parquet(
    processed_data_dir / "master_grid_dataset.parquet"
)

CN_IMPERVIOUS = 98      # standard Curve Number for impervious surfaces 
CN_GREEN = {
    "B": 61,            # standard Curve Number for green spaces on Hydrologic Soil Group B (moderate infiltration)
    "D": 80,            # standard Curve Number for green spaces on Hydrologic Soil Group D (low infiltration)
}

assert (
    master["impervious_surface_pct"]
    + master["green_space_pct"]
).eq(100).all(), (                         # validate land-cover proportions.
    "Impervious and green-space percentages must sum to 100."
)

master["CN_green"] = (
    master["hydrologic_soil_group"]
    .map(CN_GREEN)                         # map HSG to corresponding green space Curve Number
)

master["CN_impervious"] = CN_IMPERVIOUS    # assign standard Curve Number for impervious surfaces to all records

# Composite baseline Curve Number (AMC II).
master["CN_II"] = (
    master["impervious_surface_pct"] * master["CN_impervious"]
    + master["green_space_pct"] * master["CN_green"]
) / 100                                   # compute weighted average Curve Number based on land-cover proportions

print("✓ 4.1 complete: Baseline Curve Number (CN_II) computed")

master[
    [
        "hydrologic_soil_group",
        "impervious_surface_pct",
        "green_space_pct",
        "CN_green",
        "CN_II",
    ]
].head()

✓ 4.1 complete: Baseline Curve Number (CN_II) computed


,hydrologic_soil_group,impervious_surface_pct,green_space_pct,CN_green,CN_II
0,D,40,60,80,87.2
1,D,40,60,80,87.2
2,D,40,60,80,87.2
3,D,40,60,80,87.2
4,D,40,60,80,87.2


### 4.2 Antecedent Moisture Condition (AMC) and Dynamic Curve Number

Adjust each grid cell's baseline Curve Number (`CN_II`) to reflect antecedent moisture conditions prior to each rainfall event.

1. Classify AMC using season-specific 5-day rainfall thresholds
2. Convert the baseline Curve Number (`CN_II`) into a dynamic Curve Number (`CN`)
3. Use the dynamic Curve Number in subsequent runoff calculations

Dry conditions (AMC I):

$$
CN_I = \frac{4.2 \times CN_{II}}
{10 - 0.058 \times CN_{II}}
$$

Average conditions (AMC II):

$$
CN = CN_{II}
$$

Saturated conditions (AMC III):

$$
CN_{III} = \frac{23 \times CN_{II}}
{10 + 0.13 \times CN_{II}}
$$

The resulting dynamic Curve Number (`CN`) reflects changing runoff potential under dry, average, and saturated catchment conditions.

In [ ]:
master["date"] = pd.to_datetime(master["date"])   

master = (
    master
    .sort_values(["location_id", "date"])   # sort by location and date for proper temporal sequencing
    .reset_index(drop=True)                    
)

master["P_event"] = (
    master["storm_event_total_rain"]       # use total daily rainfall as the event rainfall for AMC classification
    .astype(float)                     
)

# Antecedent Moisture Condition (AMC) thresholds.
amc_thresholds = {
    "dormant": {                           # thresholds for dormant season (November to April)
        "I_max": 13.0,                   
        "II_max": 28.0,       
    },
    "growing": {                           # thresholds for growing season (May to October)
        "I_max": 36.0,
        "II_max": 53.0,
    },
}

def classify_amc(season, rain_5d):        

    season = str(season).strip().lower()     

    thresholds = amc_thresholds.get(
        season,                            
        amc_thresholds["dormant"]
    )

    if rain_5d < thresholds["I_max"]:      
        return "I"

    if rain_5d <= thresholds["II_max"]:
        return "II"

    return "III" 

# Classify antecedent moisture condition.
master["AMC_class"] = [
    classify_amc(season, rain_5d)   
    for season, rain_5d in zip(
        master["season"],
        master["rain_5d"]
    )
]

# Baseline Curve Number (AMC II).
cn_ii = master["CN_II"].astype(float)

# AMC adjustment equations.
cn_i = (
    4.2 * cn_ii
) / (
    10 - 0.058 * cn_ii
)

cn_iii = (
    23 * cn_ii
) / (
    10 + 0.13 * cn_ii
)

# Dynamic Curve Number.
master["CN"] = np.select(
    [
        master["AMC_class"].eq("I"),        
        master["AMC_class"].eq("III"),      
    ],
    [
        cn_i,
        cn_iii,
    ],
    default=cn_ii,
)

print("✓ 4.2 complete: AMC classification and dynamic CN computed")

master[
    [
        "location_id",
        "date",
        "season",
        "rain_5d",
        "AMC_class",
        "CN_II",
        "CN",
    ]
].head()

✓ 4.2 complete: AMC classification and dynamic CN computed


,location_id,date,season,rain_5d,AMC_class,CN_II,CN
0,OXF_01,2000-01-01,dormant,0.2,I,87.2,74.101651
1,OXF_01,2000-01-02,dormant,0.4,I,87.2,74.101651
2,OXF_01,2000-01-03,dormant,8.0,I,87.2,74.101651
3,OXF_01,2000-01-04,dormant,9.0,I,87.2,74.101651
4,OXF_01,2000-01-05,dormant,9.0,I,87.2,74.101651


### 4.3 Potential Maximum Retention (S)

Compute the potential maximum water retention (`S`) for each grid cell using the dynamic Curve Number (`CN`) derived in Section 4.2.

$$
S = \frac{25400}{CN} - 254
$$

Where:

- `S` is the potential maximum retention (mm)
- `CN` is the dynamic Curve Number
- Higher `CN` values produce lower retention capacity and greater runoff potential

In [ ]:
master["S"] = (
    25400.0 / master["CN"]
) - 254.0

assert (master["S"] >= 0).all()           # validate that potential maximum retention (S) is non-negative for all records

print("✓ 4.3 complete: Potential maximum retention (S) computed")

master[["CN", "S"]].head()

✓ 4.3 complete: Potential maximum retention (S) computed


,CN,S
0,74.101651,88.77239
1,74.101651,88.77239
2,74.101651,88.77239
3,74.101651,88.77239
4,74.101651,88.77239


### 4.4 Runoff Depth and Runoff Coefficient

Estimate direct runoff depth using the Soil Conservation Service (SCS) Curve Number method.

Let $P$ denote the event rainfall depth and $S$ the potential maximum retention derived in Section 4.3. Assume the initial abstraction is

$$
I_a = 0.2S
$$

Then runoff depth $Q$ is defined by

$$
Q =
\begin{cases}
\dfrac{(P-I_a)^2}{P-I_a+S}, & P > I_a \\
0, & P \le I_a
\end{cases}
$$

Substituting $I_a = 0.2S$ gives

$$
Q = \frac{(P-0.2S)^2}{P+0.8S}
$$

The runoff coefficient is then

$$
C = \frac{Q}{P}
$$

Where:

- $C$ is the runoff coefficient (dimensionless)
- $Q$ is the runoff depth (mm)
- $P$ is the event rainfall depth (mm)
- $I_a$ is the initial abstraction (mm)
- $S$ is the potential maximum retention (mm)

In [ ]:
# Event rainfall depth (mm).
P = master["P_event"].astype(float)

# Potential maximum retention (mm).
S = master["S"].astype(float)

# Initial abstraction (mm).
Ia = 0.2 * S                                   

# SCS-CN runoff depth (mm).
master["runoff_depth_mm"] = np.where(
    P > Ia,
    ((P - Ia) ** 2) / (P + 0.8 * S),
    0.0,                                        # runoff is zero if event rainfall does not exceed initial abstraction
)

# Runoff coefficient (dimensionless).
master["runoff_coefficient_C"] = np.where(      
    P > 0,
    master["runoff_depth_mm"] / P,
    0.0,
)

# Physical validity check.
assert (
    (master["runoff_coefficient_C"] >= 0)       # validate that runoff coefficient is non-negative for all records
    & (master["runoff_coefficient_C"] <= 1)     # validate that runoff coefficient does not exceed 1 for all records
).all(), "Runoff coefficient must remain between 0 and 1."

print("✓ 4.4 complete: Runoff depth and runoff coefficient computed")

master[
    [
        "P_event",
        "S",
        "runoff_depth_mm",
        "runoff_coefficient_C",
    ]
].head()

✓ 4.4 complete: Runoff depth and runoff coefficient computed


,P_event,S,runoff_depth_mm,runoff_coefficient_C
0,0.2,88.77239,0.0,0.0
1,0.2,88.77239,0.0,0.0
2,7.6,88.77239,0.0,0.0
3,1.0,88.77239,0.0,0.0
4,0.0,88.77239,0.0,0.0


### 4.5 Flood Hazard Index and Susceptibility Classification

Construct a composite Flood Hazard Index (FHI) by combining hydrological, meteorological, and topographic controls on flood generation.

### 4.5.1 Hazard Index Components

The following normalized predictors are used:

- `runoff_coefficient_C` representing runoff generation potential
- `max_1h_rain` representing short-duration rainfall intensity
- `slope` transformed into ponding susceptibility (`1 - normalized slope`)

All variables are normalized to the range [0, 1] using min-max scaling.

The Flood Hazard Index is then calculated as:

$$
FHI = 0.45C + 0.35I + 0.20F
$$

Where:

- $C$ = normalized runoff coefficient
- $I$ = normalized maximum 1-hour rainfall intensity
- $F$ = ponding susceptibility

Greater weight is assigned to runoff generation and rainfall forcing, while terrain controls are retained as secondary modifiers of flood accumulation.

River proximity is retained as contextual metadata and is not included directly in the hazard index.

### 4.5.2 Flood Susceptibility Classification

Flood susceptibility classes are derived using quantile-based thresholds applied to the Flood Hazard Index.

- `Low`: $FHI \le q_{33}$
- `Medium`: $q_{33} < FHI \le q_{66}$
- `High`: $FHI > q_{66}$

Days with no recorded rainfall and no generated runoff are assigned directly to the `Low` susceptibility class.

In [ ]:
def min_max_norm(series):
    s = series.astype(float)

    if s.max() == s.min():
        return pd.Series(0.0, index=s.index)

    return (s - s.min()) / (s.max() - s.min())


# Normalized hazard components.
master["norm_runoff_c"] = min_max_norm(
    master["runoff_coefficient_C"]
)

master["norm_rainfall_intensity"] = min_max_norm(
    master["max_1h_rain"]
)

master["norm_slope"] = min_max_norm(
    master["slope"]
)

master["ponding_susceptibility"] = (
    1.0 - master["norm_slope"]
)

# Contextual flood metadata.
master["river_context_flag"] = (          
    master["river_proximity_flag"].astype(float)  
)

# Composite Flood Hazard Index.
master["flood_hazard_index"] = (
    0.45 * master["norm_runoff_c"]
    + 0.35 * master["norm_rainfall_intensity"]
    + 0.20 * master["ponding_susceptibility"]
)

# Quantile thresholds.
q33 = master["flood_hazard_index"].quantile(0.33)
q66 = master["flood_hazard_index"].quantile(0.66)

master["flood_susceptibility_class"] = np.select(
    [
        master["flood_hazard_index"] <= q33,
        master["flood_hazard_index"] <= q66,
    ],
    [
        "Low",
        "Medium",
    ],
    default="High",
)

# Dry-day override.
dry_days = (
    (master["P_event"] <= 0)
    & (master["runoff_depth_mm"] <= 0)
)

master.loc[
    dry_days,
    "flood_susceptibility_class"
] = "Low"

print("✓ 4.5 complete: Flood Hazard Index and susceptibility classes created")

print("\nClass counts:")
print(
    master["flood_susceptibility_class"]
    .value_counts(dropna=False)
)

print("\nFlood Hazard Index summary:")
print(
    master["flood_hazard_index"]
    .describe()
)

master[
    [
        "P_event",
        "max_1h_rain",
        "runoff_coefficient_C",
        "ponding_susceptibility",
        "flood_hazard_index",
        "flood_susceptibility_class",
    ]
].head()

✓ 4.5 complete: Flood Hazard Index and susceptibility classes created

Class counts:
flood_susceptibility_class
Low       400012
High      120362
Medium     87450
Name: count, dtype: int64

Flood Hazard Index summary:
count    607824.000000
mean          0.148501
std           0.057561
min           0.000000
25%           0.112941
50%           0.158729
75%           0.183285
max           0.871479
Name: flood_hazard_index, dtype: float64


,P_event,max_1h_rain,runoff_coefficient_C,ponding_susceptibility,flood_hazard_index,flood_susceptibility_class
0,0.2,0.2,0.0,0.259366,0.053589,Low
1,0.2,0.2,0.0,0.259366,0.053589,Low
2,7.6,2.6,0.0,0.259366,0.074177,Low
3,1.0,1.0,0.0,0.259366,0.060452,Low
4,0.0,0.0,0.0,0.259366,0.051873,Low


### 4.6 Output Formatting
Standardize numeric precision for export and reporting.


In [ ]:
hydro_cols = [
    "CN",
    "S",
    "runoff_depth_mm",
    "runoff_coefficient_C",
    "ponding_susceptibility",
    "flood_hazard_index",
]

master[hydro_cols] = master[hydro_cols].round(2)   # round hydrological feature columns to 2 decimal places for better readability

print("✓ Hydrological feature precision standardized (2 d.p.)")

✓ Hydrological feature precision standardized (2 d.p.)


### 4.6 Hydrological Dataset Export

In [ ]:
engineered_data_dir.mkdir(parents=True, exist_ok=True)

feature_cols = [
    "location_id",
    "date",

    "CN_II",
    "AMC_class",
    "CN",

    "P_event",
    "max_1h_rain",

    "S",
    "runoff_depth_mm",
    "runoff_coefficient_C",

    "slope",
    "ponding_susceptibility",

    "flood_hazard_index",
    "flood_susceptibility_class",
]

available_cols = [c for c in feature_cols if c in master.columns]
output_file = engineered_data_dir / "hydrological_features.csv"
master[available_cols].to_csv(output_file, index=False)   

print(f"✓ Saved compact Section 4 feature file: {output_file}")
print(f"Columns saved ({len(available_cols)}): {available_cols}")

✓ Saved compact Section 4 feature file: c:\Users\Home\OneDrive\Python Projects\sponge-metrics\data\engineered\hydrological_features.csv
Columns saved (14): ['location_id', 'date', 'CN_II', 'AMC_class', 'CN', 'P_event', 'max_1h_rain', 'S', 'runoff_depth_mm', 'runoff_coefficient_C', 'slope', 'ponding_susceptibility', 'flood_hazard_index', 'flood_susceptibility_class']
